> **Prerequisites Note**
>
> This notebook assumes you have already executed the previous notebooks in this repository.
> The following core components are assumed to be initialized and available in your environment:
> - `llm` (ChatOpenAI / OpenRouter)
> - `embedding_model` (HuggingFace)
> - `vector_db` (Chroma)
> - `base_retriever`
>
> We intentionally reuse these objects to focus purely on conversational memory and document chains.

In [16]:
import os
import sys
from pathlib import Path

# Disable LangSmith tracing for this notebook
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_API_KEY"] = ""

# Ensure we can import from src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Restore state from Notebook 02 (Connect to existing VectorDB & LLM)
llm = ChatOpenAI(
    model=settings.MODEL_NAME,
    api_key=settings.OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

embedding_model = HuggingFaceEmbeddings(
    model_name=settings.EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

VECTOR_DB_PATH = PROJECT_ROOT / settings.VECTOR_DB_PATH
vector_db = Chroma(
    persist_directory=str(VECTOR_DB_PATH),
    embedding_function=embedding_model,
)

print("✅ Successfully connected to existing LLM and Vector Database!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5874.06it/s]


✅ Successfully connected to existing LLM and Vector Database!


# Part 1: LangChain Messages

### 1. Learning Objectives
- Understand the fundamental building blocks of conversation in LangChain 1.3+.
- Distinguish between `SystemMessage`, `HumanMessage`, and `AIMessage`.
- Learn how to structure a basic conversation history.

### 2. Concept Explanation
In modern LangChain (LCEL), conversations are not plain strings; they are structured sequences of **Message** objects. 
- **`SystemMessage`**: Sets the persona, behavior, and strict instructions for the AI.
- **`HumanMessage`**: Represents the user's input or questions.
- **`AIMessage`**: Represents the model's previous responses, used to maintain context.

### 3. Architecture Diagram (ASCII)
```text
[ SystemMessage ] -- "You are a helpful assistant."
       |
[ HumanMessage  ] -- "What is overfitting?"
       |
[  AIMessage    ] -- "Overfitting is when a model..."
       |
[ HumanMessage  ] -- "How can I prevent it?"
```

### 4. Why This Matters
Modern LLMs (like GPT-4 and Claude) are natively fine-tuned on this exact "chat" format. By passing strongly-typed message objects instead of a massive concatenated string, the model clearly understands who said what, reducing confusion and hallucination.

### 5. Best Practices
- **Always use a SystemMessage**: It anchors the model's behavior and is highly resistant to prompt injection.
- **Keep roles strict**: Never put AI responses inside a `HumanMessage`.

### 6. Common Mistakes
- **Passing raw strings**: Trying to pass a raw string to a chat model instead of a list of messages. While LangChain can sometimes implicitly cast it, explicit message objects are required for memory.
- **Reversing roles**: Accidentally assigning user input to an `AIMessage`.

### 7. Key Takeaways
- `langchain_core.messages` provides the standard data structures for all Chat Models.
- A conversation is simply a Python list of these message objects.

### 8. Mini Quiz
1. **Which message type is best for setting the AI's persona?**
   *Answer: `SystemMessage`*
2. **True or False: An LLM treats a concatenated string the exact same way as a structured list of Message objects.**
   *Answer: False. Structured messages map directly to the provider's native API roles (e.g., OpenAI's system/user/assistant roles).*
3. **What message type should be used for the historical answers generated by the model?**
   *Answer: `AIMessage`*

In [17]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

In [18]:
conversation = [
    SystemMessage(
        content="You are a helpful Machine Learning assistant."
    ),

    HumanMessage(
        content="What is overfitting?"
    ),

    AIMessage(
        content="Overfitting occurs when a model memorizes the training data and fails to generalize."
    ),

    HumanMessage(
        content="How can I prevent it?"
    )
]

In [19]:
for message in conversation:
    print("=" * 50)
    print(type(message).__name__)
    print(message.content)

SystemMessage
You are a helpful Machine Learning assistant.
HumanMessage
What is overfitting?
AIMessage
Overfitting occurs when a model memorizes the training data and fails to generalize.
HumanMessage
How can I prevent it?


# Part 2: ChatPromptTemplate + MessagesPlaceholder

### 1. Learning Objectives
- Learn to construct dynamic prompts that include variable chat history.
- Understand the role of `MessagesPlaceholder` in prompt templates.
- Safely inject historical context into a new LLM invocation.

### 2. Concept Explanation
A **`ChatPromptTemplate`** allows you to define a reusable structure for your conversation. When you have a dynamic length of past messages (chat history), you cannot hardcode them into the template. 

**`MessagesPlaceholder`** acts as a variable that unpacks a list of `Message` objects directly into the prompt sequence.

### 3. Architecture Diagram (ASCII)
```text
ChatPromptTemplate
 ├── ("system", "You are a helpful assistant.")
 │
 ├── MessagesPlaceholder(variable_name="chat_history") 
 │    └─> [Unpacks HumanMessage, AIMessage, HumanMessage...]
 │
 └── ("human", "{input}")
```

### 4. Why This Matters
In a real application, the chat history grows with every turn. `MessagesPlaceholder` provides a safe, structural way to inject an unbounded list of previous messages without breaking the template or risking prompt injection via string formatting.

### 5. Best Practices
- **Placement**: Place the `MessagesPlaceholder` *after* the `SystemMessage` but *before* the final `HumanMessage` (the current question).
- **Naming convention**: Consistently use a clear variable name like `"chat_history"` across your chains.

### 6. Common Mistakes
- **Forgetting the placeholder variable**: Defining `MessagesPlaceholder(variable_name="history")` but passing `{"chat_history": [...]}` to the `.invoke()` method, resulting in a missing variable error.
- **Using string variables for history**: Using `{chat_history}` inside a string template instead of using `MessagesPlaceholder`, which flattens the rich message objects into raw text.

### 7. Key Takeaways
- `ChatPromptTemplate` cleanly separates static instructions from dynamic data.
- `MessagesPlaceholder` is the native LangChain 1.3+ mechanism for inserting a sequence of messages into a template.

### 8. Mini Quiz
1. **Where should `MessagesPlaceholder` typically be positioned in a ChatPromptTemplate?**
   *Answer: Between the system prompt and the final human input.*
2. **What happens if you pass a string instead of a list of Messages to a `MessagesPlaceholder`?**
   *Answer: LangChain will throw a validation error, as it expects a list of Message objects.*
3. **Why is `MessagesPlaceholder` preferred over injecting history as a string?**
   *Answer: It preserves the explicit role delineations (System, Human, AI) when sent to the LLM provider.*

In [20]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="What is overfitting?"),
    AIMessage(content="Overfitting is when a model memorizes the training data.")
]

In [21]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Machine Learning assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

In [22]:
formatted_prompt = prompt.invoke({
    "chat_history": chat_history,
    "input": "How can I prevent it?"
})

# Part 3: ChatMessageHistory

### 1. Learning Objectives
- Understand the role of `ChatMessageHistory` and `InMemoryChatMessageHistory` in LangChain.
- Learn why we need a dedicated class to manage conversations instead of a plain Python list.
- Master the basic API for adding, retrieving, and clearing messages.
- Understand the Single Responsibility Principle as applied to memory storage.

### 2. Motivation

بص يا هندسة، في الأجزاء اللي فاتت إحنا عرفنا إن المحادثة بتتخزن على هيئة `List[BaseMessage]`. يعني شوية messages ورا بعض بتتبعت للـ Model. 

السؤال هنا بقى: مين اللي هيدير الـ List دي؟ مين اللي هيخزنها، يضيف عليها، أو يمسحها لو احتجنا؟ 
ممكن تقول لي: "طب ما نستخدم Python list عادية ونعمل `.append()` وخلاص!" 
نظرياً ده ينفع لو بتعمل سكريبت صغير أو بتجرب حاجة سريعة، لكن في الـ production الموضوع أعقد بكتير، وهنا بيجي دور الـ `ChatMessageHistory`.

### 3. Concept Explanation

خلينا نتخيل إن الـ `InMemoryChatMessageHistory` ده عبارة عن الـ **Conversation Notebook** أو الـ **Conversation Manager** بتاعك. هو كلاس كل شغلته في الحياة إنه يمسك ريكورد المحادثة ويحتفظ بيه. 

هنا بنطبق مبدأ مهم جداً في هندسة البرمجيات اسمه **Single Responsibility Principle (SRP)**. الـ Prompt Template مسؤوليته يجهز النص، والـ LLM مسؤوليته يفكر ويرد، أما الـ `ChatMessageHistory` فمسؤوليته الوحيدة هي إدارة وتخزين الرسايل. هو ده المدير المسؤول عن الأرشيف بتاع المحادثة.

### 4. Why Not Use a Plain Python List?

أكيد بتسأل نفسك، ليه أوجع دماغي بكلاس جديد طالما ممكن أستخدم `[]`؟

1. **Standard Interface**: الـ `ChatMessageHistory` بيورث من حاجة اسمها `BaseChatMessageHistory`. ده بيحط standard أو معيار موحد لكل أنواع الميموري في LangChain.
2. **Abstraction**: إنت بتفصل الـ logic بتاع الكود عن طريقة التخزين نفسها. الكود بتاعك مش محتاج يعرف إنت مخزن الداتا إزاي.
3. **Swappable Implementations**: لأننا شغالين بـ Interface ثابت، نقدر في المستقبل نغير الـ `InMemory` ده ونستخدم قواعد بيانات حقيقية (زي Redis، PostgreSQL، أو MongoDB) من غير ما نغير ولا سطر في الـ business logic بتاع الـ agent. الكلاس العادي مش هيقدر يعمل connection مع قاعدة بيانات ويخزن فيها الرسايل!

### 5. Architecture

```text
User 
  ↓ 
[ InMemoryChatMessageHistory ] 
  ↓ (Stores and manages Messages)
session_history.messages 
  ↓ (List of BaseMessage objects)
[ MessagesPlaceholder ]
  ↓ (Unpacks messages inside the template)
[ ChatPromptTemplate ]
  ↓ 
[ LLM ]
```

### 6. Code Implementation

يلا بينا نشوف الكود. هنبدأ بإننا نعمل import للـ `InMemoryChatMessageHistory` ونبدأ نضيف رسايل ونتعامل معاها.

In [23]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# Create an instance of our Conversation Manager
session_history = InMemoryChatMessageHistory()

# Let's add some messages manually
session_history.add_user_message("Hello, what is Machine Learning?")
session_history.add_ai_message("Machine Learning is a subfield of AI focused on building systems that learn from data.")
session_history.add_user_message("Can you give me an example?")

زي ما إنت شايف، استخدمنا `.add_user_message()` و `.add_ai_message()` عشان نضيف الرسايل. الـ functions دي بتسهل علينا بدل ما نعمل instantiate لـ `HumanMessage` و `AIMessage` بإيدينا في كل مرة ونعمل append.

دلوقتي، خلينا نشوف الأرشيف بتاعنا متخزن جواه إيه. عشان نقرأ الداتا اللي جوه، بننده على الـ property اللي اسمها `messages`.

In [24]:
# Retrieve all stored messages
stored_messages = session_history.messages

for msg in stored_messages:
    print(f"{type(msg).__name__}: {msg.content}")

HumanMessage: Hello, what is Machine Learning?
AIMessage: Machine Learning is a subfield of AI focused on building systems that learn from data.
HumanMessage: Can you give me an example?


ولو حبينا نصفر المحادثة ونبدأ من جديد (مثلاً لو اليوزر داس على زرار New Chat في الويب سايت)، الكلاس بيوفرلنا function سهلة جداً اسمها `clear()`.

In [25]:
# Clear the conversation history
session_history.clear()

print(f"Number of messages after clear: {len(session_history.messages)}")

Number of messages after clear: 0


### 7. Relationship with MessagesPlaceholder

نقطة مهمة جداً عايزك تركز فيها: لما تيجي تباصي الهيستوري ده للـ `MessagesPlaceholder` جوه الـ `ChatPromptTemplate`، إنت **لازم** تباصي `session_history.messages` ومش `history` نفسه. 

ليه؟ 
لأن الـ `MessagesPlaceholder` متبرمج إنه يستقبل `List[BaseMessage]`. الـ `history` عبارة عن Object أو كلاس بيدير الرسايل، إنما `session_history.messages` هي الـ list الفعلية اللي جواها الرسايل. لو بصيت الـ object نفسه، الـ template هيضرب منك Error ويقولك أنا مش فاهم إيه الكلاس ده.

### 8. Production Discussion

الـ `InMemoryChatMessageHistory` ده ممتاز جداً وإحنا بنجرب وبنكتب الكود في الـ Jupyter Notebook (Development / Prototyping)، لكنه **غير صالح للـ Production** نهائياً لو عندك سيستم حقيقي. 

ليه؟
- **RAM Storage**: هو بيخزن كل حاجة في الـ Memory (الرامات) بتاعة السيرفر. لو عندك آلاف اليوزرز بيكلموا الـ Agent بتاعك، الرامات هتتملي بسرعة جداً والسيرفر هيقع.
- **Data Loss**: لو السيرفر عمل Restart لأي سبب، أو عملت deploy لنسخة جديدة من الكود، كل المحادثات هتطير وهترجع صفر. 

في الـ Production، بنستخدم الـ persistent implementations (اللي بتخزن في الداتا بيز) زي `RedisChatMessageHistory` أو `PostgresChatMessageHistory` عشان نحافظ على الداتا حتى لو السيرفر فصل.

### 9. Best Practices

- **Use the Abstract Interface**: دايماً خلي الـ Type Hinting بتاعك في الـ functions هو `BaseChatMessageHistory` عشان لو غيرت الداتا بيز في المستقبل، الكود يفضل شغال وميضربش.
- **Use the helper methods**: استخدم `add_user_message()` و `add_ai_message()` بدل ما تعمل `history.add_message(HumanMessage(...))` عشان الكود يكون أنضف وأسهل في القراية.
- **Plan for storage early**: متعتمدش على الـ InMemory كتير في بناء الـ Architecture بتاعتك لو عارف إنك هتحتاج Data persistence قريب.

### 10. Common Mistakes

- **Passing the history object directly**: إنك تحاول تباصي الـ `history` للـ prompt بدل `session_history.messages`. ده أشهر خطأ بيقع فيه المبتدئين وبيجيب validation error فوراً.
- **Assuming it's production-ready**: إنك ترفع الكود بتاعك على السيرفر وإنت مستخدم `InMemoryChatMessageHistory` وتتفاجئ إن المحادثات بتاعت اليوزرز بتتمسح كل كام ساعة أو إن الرامات اتملت.

### 11. Key Takeaways

- `ChatMessageHistory` is the dedicated component responsible for managing conversation state.
- It applies the Single Responsibility Principle by abstracting storage logic away from the LLM and Prompts.
- Always pass `session_history.messages` (the list) to the `MessagesPlaceholder`, not the history object itself.
- `InMemoryChatMessageHistory` is strictly for prototyping and testing; production requires a database-backed history class.

### 12. Mini Exercise

راجع المعلومات دي مع نفسك عشان تتأكد إنك هضمت الفكرة صح:

1. **ليه بنفضل نستخدم `ChatMessageHistory` بدل الـ Python List العادية لتخزين الرسايل؟ اذكر سببين على الأقل.**
2. **لو السيرفر عمل ريستارت، إيه اللي هيحصل للرسايل المتخزنة في `InMemoryChatMessageHistory`؟ وليه؟**
3. **لو عايز تفضي المحادثة بالكامل عشان اليوزر يبدأ شات جديد، إيه الـ method اللي هتستخدمها؟**
4. **لما تيجي تستخدم `MessagesPlaceholder` جوا الـ Prompt، هل هتباصي `history` ولا `session_history.messages`؟ وليه؟**

# Part 4: History-Aware Retriever

## 1. Learning Objectives

By the end of this section, you will be able to:
- Understand the fundamental limitation of standard vector retrieval in multi-turn conversations.
- Grasp the concept of **Query Rewriting** (Contextualization).
- Implement a `History-Aware Retriever` using modern LCEL APIs.
- Evaluate the tradeoffs (latency, cost, accuracy) of adding an extra LLM call to your retrieval pipeline.

## 2. The Remaining Problem

In Part 3, we successfully implemented `ChatMessageHistory` to give our LLM a memory of past interactions. If the user asks "Who created LangChain?" and then follows up with "When was it released?", the LLM understands that "it" refers to LangChain because the entire chat history is passed into the prompt.

**However, this only solves half the problem.**

In a Retrieval-Augmented Generation (RAG) system, the LLM is not the only component that needs context. The **Retriever** (which searches the Vector Database for relevant documents) also needs to know what the user is talking about. 

While the LLM sees the full conversation, the standard Retriever is completely blind to it.

## 3. Normal Retriever Pipeline

To understand why this fails, look at the standard RAG pipeline:

```text
User Question  →  Retriever  →  Vector DB  →  Relevant Documents  →  LLM
```

When a user asks a follow-up question, the standard Retriever takes the **exact string** the user typed and converts it into an embedding vector to search the database. It does not read the `ChatMessageHistory`. It searches purely based on semantic similarity to the latest user input.

## 4. Why Retrieval Breaks

Let's look at realistic scenarios where a standard Retriever completely fails.

**Scenario A: The Pronoun Trap**
> **User**: What is LangChain?
> *(Retriever searches "What is LangChain", finds chunks, LLM answers).*
> **Assistant**: LangChain is a framework for developing applications powered by language models.
> **User**: Who created it?

The Retriever searches the Vector Database for the string *"Who created it?"*. Because there is no semantic overlap between "it" and "LangChain" in the vector space, the database returns irrelevant chunks (perhaps a document about who created Python, or who created a specific machine learning algorithm). The LLM receives useless context and fails to answer.

**Scenario B: The Implicit Comparison**
> **User**: Compare LangChain and LlamaIndex.
> **Assistant**: *(Provides a detailed comparison).*
> **User**: Which one is easier for beginners?

The Retriever searches for *"Which one is easier for beginners?"*. Again, it retrieves completely random chunks about "beginners" because the core entities (LangChain and LlamaIndex) are missing from the search query.

## 5. Query Rewriting

How do we solve this? We cannot simply concatenate the entire chat history into a massive search query. Vector databases rely on dense, specific semantic meaning. Searching the database with 50 lines of conversational history introduces extreme ambiguity and noise, degrading retrieval quality and blowing past token limits.

The engineering solution is **Query Rewriting** (or Contextualization).

Before the user's question ever reaches the Retriever, we intercept it. We pass the `ChatMessageHistory` AND the latest user question to a **small, fast LLM**. We ask this LLM to act as a translator: *"Read this conversation, and rewrite the user's latest question into a standalone query that makes sense on its own."*

The LLM translates *"When was it released?"* into *"When was LangChain released?"*. 

We then pass this dense, explicit, standalone query to the Retriever.

## 6. Architecture

Here is the architecture of a History-Aware Retriever:

```text
[ Chat History ] + [ User Question ]
            |
            v
   [ Query Rewriter (LLM) ]  ← Translates question using context
            |
            v
 [ "When was LangChain released?" ]
            |
            v
      [ Retriever ]          ← Searches with perfect context
            |
            v
     [ Vector DB ] 
            |
            v
  [ Relevant Documents ]     ← Passed to the final generation LLM
```

## 7. Build it with LCEL

LangChain still provides `create_history_aware_retriever` as a convenience helper. In this notebook we intentionally implement the same behavior manually using LCEL to understand the internal architecture.

We will use:
1. **`StrOutputParser`**: To extract the standalone question string from the rewriter LLM.
2. **`RunnablePassthrough`**: To pass the original dictionary forward and assign the new standalone question.
3. **`RunnableBranch`**: To add logic (e.g., skip rewriting if the chat history is empty).

## 8. Imports

Let's import the necessary components for our History-Aware Retriever.

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch


## 9. Build the Retriever

First, we need our base retriever. We will reuse the `vector_db` created earlier in this notebook.

In [27]:
# Convert the vector database into a standard retriever
# We configure it to return the top 3 most relevant chunks
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

## 10. Build the Contextualization Prompt

Now we build the prompt that will instruct the LLM to rewrite the user's query. 

Notice how this prompt is constructed:
- The **SystemMessage** gives strict instructions: reformulate the question, but *do not answer it*.
- The **MessagesPlaceholder** dynamically injects the past conversation (`chat_history`).
- The **HumanMessage** injects the user's ambiguous follow-up question (`input`).

In [28]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. "
    "Do NOT answer the question, just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

## 11. Create History-Aware Retriever

LangChain still provides `create_history_aware_retriever` as a convenience helper. In this notebook we intentionally implement the same behavior manually using LCEL to understand the internal architecture.
This creates a pipeline that orchestrates the query rewriting and document retrieval automatically.

We are implementing the behavior manually with LCEL for educational purposes. In production you may also use LangChain's helper `create_history_aware_retriever`.

In [29]:
# 1. The query rewriter chain (Prompt -> LLM -> String)
query_rewriter = contextualize_q_prompt | llm | StrOutputParser()

# 2. The full history-aware retriever
# It checks if chat_history is empty. If empty, it just uses the input.
# If not empty, it rewrites the query first.
history_aware_retriever = (
    RunnablePassthrough.assign(
        standalone_question=RunnableBranch(
            # Condition: If no chat history, just return the input
            (lambda x: not x.get("chat_history", []), lambda x: x["input"]),
            # Default: Rewrite the query
            query_rewriter
        )
    )
    | (lambda x: x["standalone_question"])
    | base_retriever
)

### Official LangChain Equivalent

While we implemented this manually using pure LCEL to understand the architecture, LangChain provides a convenience helper function for this exact pipeline.

**Why implement manually?**
To understand that there is no "magic". It is simply a `RunnableBranch` that checks if the chat history is empty, and a `StrOutputParser` that feeds into a retriever.

**When to use the helper?**
In production, use `create_history_aware_retriever()` to save boilerplate code and ensure forward compatibility with LangChain updates.

```python
from langchain.chains import create_history_aware_retriever

# This one-liner replaces the entire LCEL block above
helper_history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=base_retriever,
    prompt=contextualize_q_prompt
)

```

## 12. Demonstration

Let's simulate a follow-up conversation. We will provide a fake chat history where the user previously asked about "LangChain", and then ask an ambiguous follow-up question: "Who created it?".

When we invoke the `history_aware_retriever`, it will internally rewrite "Who created it?" to "Who created LangChain?", and use that to search the database.

In [30]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate past conversation
demo_chat_history = [
    HumanMessage(content="What is LangChain?"),
    AIMessage(content="LangChain is a framework for developing applications powered by language models.")
]

# The ambiguous follow-up question
ambiguous_query = "Who created it?"

# Invoke the history-aware retriever
retrieved_docs = history_aware_retriever.invoke({
    "chat_history": demo_chat_history,
    "input": ambiguous_query
})

print(f"User asked: '{ambiguous_query}'")
print(f"Retrieved {len(retrieved_docs)} documents based on the rewritten query.\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i} Snippet: {doc.page_content[:150]}...")

User asked: 'Who created it?'
Retrieved 3 documents based on the rewritten query.

Document 1 Snippet: • Built 15+ slide deck visualizing China’s $1.5 billion tutoring industry for post-grad entrance exam 
• Conducted 10+ interviews with tech execs mapp...
Document 2 Snippet: aligned mitigation strategies, enabling compliance with international standards through monitorable
outcomes.
Lexicon            Jun 2022 – Jun 2023 
...
Document 3 Snippet: 2011 – Present 
PROFESSIONAL EXPERIENCE 
        SYNOPSIS, INC., Marlborough, MA 
        Senior Application Consultant II - Synplicite  Product Sales...


*Note: Because this chain returns `Document` objects directly from the vector store, we don't see the intermediate rewritten string in the standard output. *

## 13. Production Discussion

Adding a History-Aware Retriever solves a critical accuracy issue, but it introduces major architectural tradeoffs you must account for in production:

- **Latency**: You are now making **two** sequential LLM calls for every user message. (1. Rewrite Query → 2. Retrieve Docs → 3. Generate Final Answer). This significantly increases Time to First Token (TTFT).
- **Cost**: Query rewriting consumes tokens. For high-traffic applications, this effectively doubles your LLM inference costs.
- **When to skip**: If the user's input is a completely new topic (e.g., "Forget LangChain, tell me about Docker"), the rewriting step is wasted compute. Advanced systems use a cheap classifier to decide if rewriting is necessary before routing.

## 14. Best Practices

- **Use a cheaper/faster LLM for rewriting**: You do not need GPT-4 or Claude 3.5 Sonnet to rewrite a query. Use a much faster, cheaper model (like GPT-4o-mini or Claude Haiku) specifically for the query rewriter step, and save your expensive model for generating the final answer.
- **Keep the contextualization prompt strict**: Ensure the system prompt explicitly commands the LLM *not* to answer the question, otherwise the LLM might output a full paragraph instead of a standalone query, ruining the vector search.

## 15. Common Mistakes

- **Concatenating history manually**: Attempting to bypass this pattern by just concatenating all previous messages into a single string and passing it to the Vector Database. This dilutes the semantic density of the query and destroys retrieval accuracy.
- **Forgetting the Placeholder**: Omitting `MessagesPlaceholder` in the contextualization prompt. The LLM will rewrite the query blindly without knowing the chat history, resulting in failure.

## 16. Key Takeaways

- A standard Retriever is stateless; it fails on ambiguous follow-up questions because it cannot resolve pronouns or implied context.
- A History-Aware Retriever intercepts the user's question and uses an LLM to rewrite it into a standalone query based on the chat history.
- We orchestrate this process using **pure LCEL**, combining `RunnableBranch` and `StrOutputParser` to feed the standalone question into the vector store retriever.
- This pattern drastically improves retrieval accuracy at the cost of increased latency and API usage.

## 17. Mini Exercise

1. What specific instruction in `contextualize_q_system_prompt` prevents the LLM from acting like a chatbot during the rewriting phase?
2. If you are building a Conversational RAG system with a tight budget and strict latency requirements (e.g., responses must be under 1 second), how might the History-Aware Retriever impact your architecture?
3. What is the expected behavior of the query rewriter if the user asks a completely standalone question like "What is the capital of France?" despite the chat history being about Machine Learning?

# Part 5: Building a Complete Retrieval Chain

## 1. Learning Objectives

By the end of this section, you will be able to:
- Understand the gap between retrieving raw documents and generating a final answer.
- Conceptually grasp different document processing strategies (Stuff, Map Reduce, Refine).
- Manually construct the internals of `create_stuff_documents_chain` and `create_retrieval_chain` using LCEL.
- Execute and debug a fully functional, history-aware Conversational RAG pipeline.
- Apply production best practices for prompt design and retrieval quality.
- Preview how `RunnableWithMessageHistory` removes manual `chat_history` bookkeeping for multi-user, session-based conversations.


## 2. The Missing Piece

At this point in the notebook, we have a `history_aware_retriever`. When we pass a question and chat history to it, it successfully rewrites the query and pulls relevant chunks from our Vector Database.

But look at what it returns:

```text
User  →  Retriever  →  [ Document 1, Document 2, Document 3 ]
```

A standard retriever returns a list of **raw documents**. It does *not* return an answer. If you send a list of LangChain `Document` objects back to a user in a chat interface, they will be extremely confused.

We need a component that takes these raw documents, injects them into an LLM prompt, and asks the LLM to read them and generate a conversational answer. This is the missing piece.

## 3. Understanding Document Chains

A **Document Chain** is responsible for converting raw documents into a coherent response.

- **Inputs**: It receives a user query (like `{input}`) and a list of retrieved documents (like `{context}`).
- **Responsibilities**: It formats the documents into a readable string, places them into a prompt template, and invokes the LLM.
- **Outputs**: A final generated string (the answer).

Without a Document Chain, RAG cannot exist. The retriever acts as the "eyes" searching the library, and the document chain acts as the "brain" reading the books and summarizing the answer.

## 4. Why `create_stuff_documents_chain()`?

LangChain provides several strategies for dealing with retrieved documents:

1. **Stuff**: Take all retrieved documents and "stuff" them directly into the context window of a single prompt.
2. **Map Reduce**: Pass *each* document to the LLM individually to extract an answer, then pass all the extracted answers to the LLM again to synthesize a final response.
3. **Refine**: Pass the first document to the LLM to get an initial answer, then pass that answer plus the second document to the LLM to refine it, and so on.

The **Stuff** method (implemented via `create_stuff_documents_chain`) is by far the most common in modern RAG. 
- **Advantages**: It only requires one LLM call, making it extremely fast and cheap. The LLM sees all context simultaneously, allowing it to draw cross-document connections.
- **Limitations**: It is strictly bound by the LLM's context window. If you retrieve 50 large documents, you cannot "stuff" them all without exceeding token limits.
- **When NOT to use it**: When doing massive summarization tasks across hundreds of pages where context window limits are exceeded.

## 5. Visualizing the Pipeline

Here is the internal workflow of a **Document Chain**:

```text
  [Document 1] + [Document 2] + [Document 3]
                       |
                       v
        [ Formatted Context String ]
                       |
                       v
         [ Chat Prompt Template ]
                       |
                       v
                    [ LLM ]
                       |
                       v
                [ Final Answer ]
```

When we combine the Retriever and the Document Chain, we get the **Full Retrieval Chain**:

```text
Question  →  Retriever  →  Raw Documents  →  Document Chain  →  Answer
```

## 6. Build the Prompt

Before we build the chain, we need the Prompt. 
We will construct a `ChatPromptTemplate` that includes:
- **System Prompt**: Defines the persona and strict rules (e.g., "answer only using context").
- **`{context}`**: The placeholder where our stuffed documents will go.
- **`MessagesPlaceholder`**: For our chat history.
- **Human Prompt**: The user's latest question.

In [31]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

qa_system_prompt = (
    "You are an expert Career AI Assistant. Your goal is to provide accurate, "
    "professional answers based strictly on the provided context.\n\n"
    "RULES:\n"
    "1. Answer ONLY using the retrieved context.\n"
    "2. If the answer is not in the context, say 'I do not know'.\n"
    "3. Do not hallucinate or invent information.\n\n"
    "CONTEXT:\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

print("QA Prompt built. Required inputs:", qa_prompt.input_variables)

QA Prompt built. Required inputs: ['chat_history', 'context', 'input']


## 7. Build `create_stuff_documents_chain()`

> *Note: LangChain provides `create_stuff_documents_chain` as a convenience helper. In this notebook, we intentionally implement the same behavior manually using pure LCEL to deeply understand the internal architecture.*

To build this chain, we need to:
1. Take a dictionary containing `context` (a list of Documents) and `input`.
2. Convert the list of Documents into a single string.
3. Pass the resulting dictionary into our `qa_prompt`.
4. Pass the prompt to the `llm`.
5. Parse the output into a string using `StrOutputParser`.

In [32]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs: list[Document]) -> str:
    """Extracts text and metadata from Document objects and joins them."""
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Source: {source}\nContent: {doc.page_content}")
    return "\n\n---\n\n".join(formatted)

# LCEL implementation of a Stuff Documents Chain
stuff_documents_chain = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("Stuff Documents Chain successfully built using LCEL!")

Stuff Documents Chain successfully built using LCEL!


### Official LangChain Equivalent

**Why implement manually?**
To see exactly how documents are joined into a string and injected into the prompt. It demystifies how the "stuffing" mechanism actually manipulates the prompt context.

**When to use the helper?**
Use `create_stuff_documents_chain()` in production. It automatically handles document formatting and integrates perfectly with LangChain's tracing ecosystem.

```python
from langchain.chains.combine_documents import create_stuff_documents_chain

# This replaces our manual format_docs and LCEL pipeline
helper_stuff_documents_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=qa_prompt
)

```

## 8. Test the Document Chain

Let's test the document chain in isolation. We will manually provide a mock document and a question. Notice that we bypass the retriever completely here; we are just proving that the LLM can read the injected documents and answer correctly.

In [33]:
mock_docs = [
    Document(page_content="LangChain was launched in October 2022 by Harrison Chase.")
]

test_response = stuff_documents_chain.invoke({
    "context": mock_docs,
    "input": "When was LangChain launched?",
    "chat_history": []
})

print("LLM Answer:\n", test_response)

LLM Answer:
 LangChain was launched in October 2022.


## 9. Build Retrieval Chain

> *Note: Similarly, LangChain provides `create_retrieval_chain` as a helper. We implement it manually here via LCEL to retain full control over the execution graph and debug outputs.*

The Retrieval Chain unites the Retriever and the Document Chain.

- **Inputs**: A dictionary containing `input` and `chat_history`.
- **Architecture**: It first passes these inputs to the `history_aware_retriever`. The retriever grabs the documents and assigns them to the `context` key. Then, the whole dictionary (input, chat_history, context) is passed to the `stuff_documents_chain`, which generates the `answer`.
- **Outputs**: A dictionary containing all the original inputs, the retrieved `context`, and the final `answer`.

In [34]:
# LCEL implementation of the full Retrieval Chain
retrieval_chain = (
    # 1. Fetch documents using the history-aware retriever
    RunnablePassthrough.assign(
        context=history_aware_retriever
    )
    # 2. Generate the answer using the stuffed documents
    .assign(
        answer=stuff_documents_chain
    )
)

print("Complete Retrieval Chain built successfully!")

Complete Retrieval Chain built successfully!


### Official LangChain Equivalent

**Why implement manually?**
To understand the dictionary passthrough logic. The retriever must fetch documents and assign them to the `context` key before passing the entire payload to the document chain.

**When to use the helper?**
Always use `create_retrieval_chain()` in production to wire your retriever and document chain together seamlessly.

```python
from langchain.chains import create_retrieval_chain

# This wires the history-aware retriever to the stuff documents chain
helper_retrieval_chain = create_retrieval_chain(
    retriever=history_aware_retriever,
    combine_docs_chain=stuff_documents_chain
)

```

## 10. Execute Retrieval Chain

Now let's run a realistic, multi-turn conversation end-to-end against our Vector Database.

In [35]:
from langchain_core.messages import HumanMessage, AIMessage

demo_chat_history = []

# Turn 1: A relevant query for our dataset
question_1 = "What skills are required for a Machine Learning Engineer?"
result_1 = retrieval_chain.invoke({
    "input": question_1,
    "chat_history": demo_chat_history
})

# Update history manually
demo_chat_history.append(HumanMessage(content=question_1))
demo_chat_history.append(AIMessage(content=result_1["answer"]))

# Turn 2: Ambiguous follow-up requiring History-Awareness
question_2 = "Which of those is the most important for a beginner?"
result_2 = retrieval_chain.invoke({
    "input": question_2,
    "chat_history": demo_chat_history
})

print("\n" + "="*50)
print("QUESTION 1:", question_1)
print("ANSWER 1:", result_1["answer"])
print("="*50)
print("QUESTION 2 (Follow-up):", question_2)
print("ANSWER 2:", result_2["answer"])
print("="*50)


QUESTION 1: What skills are required for a Machine Learning Engineer?
ANSWER 1: I do not know.
QUESTION 2 (Follow-up): Which of those is the most important for a beginner?
ANSWER 2: I do not know.


## 11. Debugging

When building RAG, you must constantly debug why an LLM answered a certain way. 

By inspecting the returned dictionary from our LCEL chain, we can view exactly which chunks were retrieved from the Vector Store. If the final answer is wrong, inspecting the `context` key tells you instantly if the Retriever failed (bad chunks) or if the LLM failed (hallucination).

In [36]:
print("--- 1. Inspecting Output Keys ---")
print(list(result_2.keys()))

print("\n--- 2. Inspecting Retrieved Context Quality ---")
retrieved_docs = result_2.get("context", [])
print(f"Total Documents Retrieved: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get("source", "Unknown Source")
    page = doc.metadata.get("page", "N/A")
    print(f"\n[Chunk {i}] Source: {source} (Page {page})")
    print(f"Snippet: {doc.page_content[:100]}...")

print("\n--- 3. Verifying Final Answer ---")
print(result_2["answer"])

--- 1. Inspecting Output Keys ---
['input', 'chat_history', 'context', 'answer']

--- 2. Inspecting Retrieved Context Quality ---
Total Documents Retrieved: 3

[Chunk 1] Source: d:\career-ai-agent\data\interview\Meta_ML_onsite_interview_prep.pdf (Page 4)
Snippet: 5
• Brush up on basic ML theory and algorithm details. Be comfortable with
concepts like overfitting...

[Chunk 2] Source: d:\career-ai-agent\data\interview\10-41-Essential-Machine-Learning-Interview-Questions.pdf (Page 13)
Snippet: (classification, prediction, etc.) and bring up a few examples and use 
cases.
Q25- What’s the “kern...

[Chunk 3] Source: d:\career-ai-agent\data\interview\10-41-Essential-Machine-Learning-Interview-Questions.pdf (Page 19)
Snippet: Related to the last point, most organizations hiring for machine learn-
ing positions will look for ...

--- 3. Verifying Final Answer ---
I do not know.


## 12. Production Best Practices

- **Prompt Design**: Never trust the LLM to "just know" it should use the context. Use capital letters (`CONTEXT:`, `RULES:`) and explicitly state that hallucination is forbidden.
- **Context Window Management**: Be aware of token limits. If your `chunk_size` is 1000 tokens and you retrieve `k=10` chunks, you are shoving 10,000 tokens into the prompt. Ensure your chosen model supports this.
- **Chunk Quality**: The Document Chain can only synthesize what it reads. If the text splitter cut a sentence in half during ingestion, the Document Chain will struggle to understand it.
- **Hallucination Reduction**: Adding a strict out-clause like *"If you do not know the answer, say 'I don't know'"* drastically reduces confident hallucinations.

## 13. Common Mistakes

- **Too many chunks**: Retrieving too many documents (e.g., `k=20`) can cause "Lost in the Middle" syndrome, where the LLM forgets information placed in the center of the prompt.
- **Ignoring citations**: In production, users want proof. Returning just the answer string without exposing the source documents (metadata) destroys trust.
- **Bad Prompt Formatting**: Passing raw document objects directly into a string template without properly joining their text contents.

## 14. Preview: Automatic History Management

In our demonstration above, we manually updated the `demo_chat_history` list after every LLM call using `.append()`. While this is excellent for learning how state transitions work, it is **terrible for production**.

In a real application with concurrent users, you cannot manually append messages to global variables. You need a system that automatically fetches a user's past messages from a database, injects them into the prompt, and automatically saves the new response back to the database.

LangChain solves this with **`RunnableWithMessageHistory`**.

### Architecture

```text
User Request (session_id="user_123", question="Who created it?")
       |
       v
[ RunnableWithMessageHistory ]
       |-- 1. Queries Database for "user_123" history
       |-- 2. Injects history into the LCEL Chain
       v
 [ Retrieval Chain ] (Runs exactly as we built it)
       |
       v
[ RunnableWithMessageHistory ]
       |-- 3. Intercepts the generated answer
       |-- 4. Saves Question & Answer back to the Database for "user_123"
       v
 Final Answer returned to User
```

In [37]:
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}  # Mock database (session_id -> ChatMessageHistory)

# We define a function that retrieves a ChatMessageHistory object based on a session ID
def get_session_history(session_id: str):
    # In production, this would query Redis or PostgreSQL using the session_id
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap our retrieval chain with automatic memory management
conversational_rag_chain = RunnableWithMessageHistory(
    runnable=retrieval_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

print("Production-ready Conversational RAG built! (Requires explicit session_id during invocation)")


Production-ready Conversational RAG built! (Requires explicit session_id during invocation)


d:\career-ai-agent\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 15. Mini Exercise

1. Modify `qa_system_prompt` to force the LLM to answer in the tone of a pirate, but still strictly use the context.
2. `format_docs` now includes the source filename for each chunk. Extend it further to also include the page number (`doc.metadata.get("page")`) so the LLM can cite `Source: file.pdf, Page: 3` instead of just the filename.
3. What happens if you change the retrieval strategy to Map-Reduce? How many LLM calls would be made if `k=5`?
4. In `get_session_history`, what would break if two different users invoked `conversational_rag_chain` with the same `session_id`? How would you generate safer session IDs in a real web app?


## 16. Key Takeaways

- A Retriever returns documents; a Document Chain reads those documents to generate an answer.
- The `Stuff` strategy is the most common, placing all retrieved text into a single prompt.
- By building the Retrieval Chain manually via LCEL, we ensure a clean dictionary output containing our `input`, `chat_history`, `context`, and final `answer`.
- Debugging RAG means inspecting the `context` key before blaming the LLM for a bad answer.
- `RunnableWithMessageHistory` removes the need to manually manage a `chat_history` list per user — it fetches, injects, and saves history automatically per `session_id`, which is what makes this pipeline usable by more than one concurrent user.
